## Overview

Reproduces the **SEEDA paper's synthetic scenario 1 (Table 2)**: K = 6 doses, toxicity `[0.01, 0.05, 0.15, 0.2, 0.45, 0.6]`, efficacy `[0.1, 0.35, 0.6, 0.6, 0.6, 0.6]`, θ = 0.35, 1000 trials — matching the paper's exact numbers rather than the ad hoc scenarios in `archive/benchmarks.ipynb` / `archive/benchmarks_with_original_and_fixed.ipynb` (which instead replicate UCB's PPT experiment).

Benchmarks 9 designs: `3 + 3`, `CRM`, `UCB`, two SEEDA variants (log vs. no-log exploration bonus), and four SEEDA-Plateau variants (paper-faithful, UCB's, and two corrected L1 rules — see `docs/SEEDA_PLATEAU_L1_INVESTIGATION.md` for the derivation). Reproduces the paper's Figures 1-3 and Table 2. **This is the current/active notebook.**

In the following set of experiments, we aim to generate meaningful benchmarks for the dose escalation methods we would like to study.

### Utilities

In [1]:
import numpy as np
from datetime import datetime
from doseescalation.dose_escalator import (
    CRMDoseEscalator, 
    DoseEscalatorBase,
    ThreePlusThreeDoseEscalator, 
    UCBDoseEscalator,
    SEEDADoseEscalator,
    SEEDAOriginalDoseEscalator,
    SEEDAPlateauDoseEscalator,
    SEEDAPlateauNaiveDoseEscalator,
    SEEDAPlateauTwoSidedDecoupledDoseEscalator,
    SEEDAPlateauTwoSidedDecoupledNoLogDoseEscalator
)
from doseescalation.estimator import (
    AveragingEstimator
)
from doseescalation.evaluate import (
    plot_dose_proposals, 
    plot_acc_progression,
    plot_sample_efficiency,
    plot_efficacy_and_violation,
    plot_error_rates,
    plot_dose_curves,
    plot_n_dles,
    simulate
)
from doseescalation.build_tables import (
    build_results_table,
    build_correct_dose_summary,
    build_table2,
    style_table2,
    export_table2_latex
)
from doseescalation.simulated_env import SimulatedEnv
from typing import Callable, Sequence

/opt/anaconda3/envs/ucl/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
# Get the current timestamp for saving results:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [3]:
def a_key(a):
    return f"a = {a:.1f}"

def cohort_key(cohort):
    return f"Cohort {cohort + 1}"

In [4]:
def dose_toxic_curve(dose_levels, a_hat):
    return np.power((np.tanh(dose_levels) + 1) / 2, a_hat)

def inv_dose_toxic(p_dle, a):
    return np.arctanh(2 * np.power(p_dle, 1 / a) - 1)

In [5]:
# Define hyperparameter a:
A_STAR = 1.0

# Dose-toxicity curve (Table 2 from SEEDA paper):
TOXICITY_PROBS = [0.01, 0.05, 0.15, 0.2, 0.45, 0.6]

# Dose-efficacy curve is the per-patient response ~ Bernoulli(q_dose),
# rising then plateauing (shared across all toxicity scenarios, if any):
EFFICACY_PROBS = [0.1, 0.35, 0.6, 0.6, 0.6, 0.6]

# Simulation settings:
N_LEVELS = 6
P_DLE_LEVELS = {
    A_STAR: TOXICITY_PROBS
}
DOSE_LEVELS = {
    a: [inv_dose_toxic(v, a) for v in vs] 
    for a, vs in P_DLE_LEVELS.items()
}
TTL = 0.35
N_TRIALS = 1000
COHORT_SIZE = 3

# Number of patients who showed positive response must be <= COHORT_SIZE (≈ 33% efficacy rate):
N_EFFICATE = 1

# Efficacy env: reuse SimulatedEnv with "dose levels" = indices so the curve maps
# index -> probability, giving n_efficate ~ Binomial(cohort, EFFICACY_PROBS[idx]).
EFFICACY_ENV = SimulatedEnv(list(range(N_LEVELS)), lambda i: EFFICACY_PROBS[int(i)])

# To add more algorithms, extend the following list to include them.
# Four SEEDA-Plateau variants are compared:
#   - "Paper"                        : the paper's faithful (naive) first-flat-pair L1.
#   - "UCB"                          : UCB's Plateau, entire-upper-tail-flat L1.
#   - "Two-sided, decoupled"         : UCB's tail L1 with Part B dropped (two-sided
#                                      only) and a separate small L1 coefficient.
#   - "Two-sided, decoupled, NoLog"  : same, but with the no-log exploration bonus.
# See docs/SEEDA_PLATEAU_L1_INVESTIGATION.md for why the last two recover dose 3.
ALGOS = [
    "3 + 3", "CRM", "UCB",
    "SEEDA", "SEEDA (UCB)",
    "SEEDA Plateau (Paper)", "SEEDA Plateau (UCB)",
    "SEEDA Plateau (Two-sided, decoupled)",
    "SEEDA Plateau (Two-sided, decoupled, NoLog)",
]

# UCB parameter:
UCB_COEFF = 0.1

# SEEDA and SEEDA Plateau parameters:
P_HAT = (0.02, 0.06, 0.12, 0.20, 0.30, 0.40)
Q_HAT = (0.12, 0.20, 0.30, 0.40, 0.50, 0.59)
ETA = 2
SEEDA_UCB_COEFF = 2.1
# Separate (small) confidence coefficient for the two-sided-decoupled Plateau L1
# test, decoupled from the allocation UCB coefficient above:
L1_COEFF = 0.1

# k* = lowest safe dose (p <= theta) with maximal efficacy. Safe doses are 1-4
# (idx 0-3); max efficacy 0.6 is reached at doses 3 & 4 -> k* = dose 3 (index 2).
# Toxicity MTD (highest safe dose) = dose 4 (index 3):
OPTIMAL_DOSE = 2
TOX_MTD = 3
OPTIMAL_DOSES = {a_key(A_STAR): OPTIMAL_DOSE}
CORRECT_MTDS  = {a_key(A_STAR): TOX_MTD}

# Every design is scored and highlighted against the optimal biological dose k* (dose 3):
CORRECT_DOSES = {
    a_key(a): {algo: OPTIMAL_DOSE for algo in ALGOS}
    for a in DOSE_LEVELS
}

**Scenario curves.** Dose-toxicity model $p_k(a) = (\frac{\tanh(d_k)+1}{2})^a$ for several values of the global parameter $a$, and the dose-efficacy curve. At $a = a*$ the toxicity curve passes through the Table 2 probabilities by construction (the dose levels were back-solved from them). θ is the MTD threshold; $k*$ is the optimal biological dose.

In [6]:
plot_dose_curves(
    DOSE_LEVELS[A_STAR],
    EFFICACY_PROBS,
    dose_toxic_curve,
    TTL,
    a_values=[0.5, A_STAR, 2.0, 3.0],
    a_star=A_STAR,
    optimal_dose=OPTIMAL_DOSE,
    show_fig=False,
    img_path=f"plots/{timestamp}/dose_curves.png",
)

In [7]:
def run_simulations(
    dose_escalator: DoseEscalatorBase,
    dose_levels: Sequence[float],
    dose_toxic_curve: Callable,
    cohort_size: int, 
    n_cohorts: int,
    n_efficate: int = 0,
    efficacy_env=None,
):
    env = SimulatedEnv(dose_levels, dose_toxic_curve)
    return simulate(
        cohort_sizes=[cohort_size] * n_cohorts, 
        dose_escalator=dose_escalator, 
        env=env,
        n_efficate=n_efficate,
        efficacy_env=efficacy_env
    )

### Simulations

We run 2 main sets of simulations, differing in the number of cohorts used in their experiments. 

The more realistic setting uses around 10^1 cohorts while the setting for asymptotic behaviours uses more than 10^2.

#### Realistic

In [8]:
N_REAL_COHORTS = 10

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [9]:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_REAL_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_safe_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_REAL_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

def add_simulations(dose_escalator, a, algo, n_efficate=0, n_cohorts=N_REAL_COHORTS):
    allocations, recommendations, n_dles, safe_sets = run_simulations(
        dose_escalator,
        dose_levels,
        lambda dose: dose_toxic_curve(dose, a),
        COHORT_SIZE,
        n_cohorts=n_cohorts,
        n_efficate=n_efficate,
        efficacy_env=EFFICACY_ENV
    )
    # Determine the final declared MTD for this trial:
    a_algo_rec_map[a_key(a)][algo].append(recommendations[-1])

    # Determine every cohort's allocated dose pooled across trials:
    a_algo_alloc_map[a_key(a)][algo].extend(allocations)

    # Determine the total toxicities during this trial:
    a_algo_n_dle_map[a_key(a)][algo].append(sum(n_dles))

    # Determine the recommendation at each cohort:
    for cohort, rec in enumerate(recommendations):
        cohort_algo_rec_map[a_key(a)][cohort_key(cohort)][algo].append(rec)

    # Determine the per-dose safety classification at each cohort:
    for cohort, safe in enumerate(safe_sets):
        cohort_algo_safe_map[a_key(a)][cohort_key(cohort)][algo].append(safe)

# Run the realistic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0])
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1])
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2])
        
        # SEEDA (current version):
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE)

        # SEEDA (UCB):
        seeda_original_dose_escalator = SEEDAOriginalDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_original_dose_escalator, a, ALGOS[4], N_EFFICATE)

        # SEEDA Plateau (Paper) - faithful first-flat-pair L1:
        seedapl_paper_dose_escalator = SEEDAPlateauNaiveDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_paper_dose_escalator, a, ALGOS[5], N_EFFICATE)

        # SEEDA Plateau (UCB) - entire-upper-tail-flat L1:
        seedapl_ucb_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_ucb_dose_escalator, a, ALGOS[6], N_EFFICATE)

        # SEEDA Plateau (Two-sided, decoupled) - UCB tail L1, no Part B, small L1 c:
        seedapl_twosided_dose_escalator = SEEDAPlateauTwoSidedDecoupledDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            l1_coefficient=L1_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_twosided_dose_escalator, a, ALGOS[7], N_EFFICATE)

        # SEEDA Plateau (Two-sided, decoupled, NoLog) - same but no-log exploration:
        seedapl_twosided_nolog_dose_escalator = SEEDAPlateauTwoSidedDecoupledNoLogDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            l1_coefficient=L1_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_twosided_nolog_dose_escalator, a, ALGOS[8], N_EFFICATE)

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [10]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Realistic/Recommendations/real_algo_dose_rec_proposals.png"
)

Plot the proposals made for each cohort, so that we can see the time evolution of our `DoseEscalator`'s proposals. This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [11]:
for a in DOSE_LEVELS.keys():
    correct_mtds = {
        cohort_key(cohort): CORRECT_DOSES[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    mtds = {
        cohort_key(cohort): CORRECT_MTDS[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    plot_dose_proposals(
        N_LEVELS, 
        N_TRIALS, 
        cohort_algo_rec_map[a_key(a)], 
        correct_mtds, 
        mtds=mtds,
        unit_width=100,
        title_text="MTD recommendations at each cohort",
        show_fig=False,
        img_path=f"plots/{timestamp}/Realistic/Recommendations/real_{a_key(a)}_dose_rec_proposal_progression.png"
    )

Plot the distribution of dose limiting events across all the trial runs and cohorts.

In [12]:
plot_n_dles(
    a_algo_n_dle_map, 
    img_path=f"plots/{timestamp}/Realistic/real_algo_n_dles.png"
)

Plot the distribution of dose allocations (the dose each cohort actually received), pooled across all cohorts and trial runs.

In [13]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_REAL_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Realistic/Allocations/real_algo_dose_allocations.png"
)

Recommendation accuracy progression (realistic, n = 10 cohorts): fraction of trials recommending k* at each cohort.

In [14]:
plot_acc_progression(
    N_REAL_COHORTS,
    cohort_algo_rec_map,
    CORRECT_DOSES,
    show_fig=False,
    img_path=f"plots/{timestamp}/Realistic/Recommendations/real_dose_rec_proposal_acc_progression.png"
)

Sample efficiency (Figure 3 in the SEEDA paper), realistic setting. Conforms to the paper's definitions (Section 2.2 successful recommendation, Section 5.1.3).

In [15]:
plot_sample_efficiency(
    N_REAL_COHORTS,
    COHORT_SIZE,
    cohort_algo_rec_map,
    CORRECT_DOSES,
    algos=[a for a in ALGOS if a not in ("3 + 3", "CRM")],
    show_fig=False,
    img_path=f"plots/{timestamp}/Realistic/Recommendations/real_sample_efficiency.png"
)

Efficacy per patient and safety-violation percentage (Figure 2 in the SEEDA paper), realistic setting. Conforms to the paper's definitions (Section 2.2 / Eq. (2) / Theorem 2).

In [16]:
plot_efficacy_and_violation(
    N_REAL_COHORTS,
    N_TRIALS,
    a_algo_alloc_map,
    EFFICACY_PROBS,
    TOXICITY_PROBS,
    TTL,
    img_path=f"plots/{timestamp}/Realistic/real_efficacy_and_violation.png"
)

Type I / Type II error rates (Figure 1 in the SEEDA paper), realistic setting. Conforms to the paper's definitions (supplementary Section J, Lemma 1/2).

In [17]:
plot_error_rates(
    N_REAL_COHORTS,
    cohort_algo_safe_map,
    TOXICITY_PROBS,
    TTL,
    show_fig=False,
    img_path=f"plots/{timestamp}/Realistic/real_error_rates.png"
)

Make a table with the metrics.

In [18]:
# Long-format (scenario, algorithm, dose) table feeding the correct-dose summary below.
# Table 2 further down uses the raw maps directly, not this table:
real_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, OPTIMAL_DOSES, N_LEVELS, ALGOS
)

correct_summary = build_correct_dose_summary(real_table, CORRECT_MTDS)
print("Correct-dose summary")
display(correct_summary)

Correct-dose summary


Correct dose  Is MTD  \
Scenario Algorithm                                                           
a = 1.0  3 + 3                                                   3   False   
         CRM                                                     3   False   
         UCB                                                     3   False   
         SEEDA                                                   3   False   
         SEEDA (UCB)                                             3   False   
         SEEDA Plateau (Paper)                                   3   False   
         SEEDA Plateau (UCB)                                     3   False   
         SEEDA Plateau (Two-sided, decoupled)                    3   False   
         SEEDA Plateau (Two-sided, decoupled, NoLog)             3   False   

                                                      Correct dose rec %  \
Scenario Algorithm                                                         
a = 1.0  3 + 3                                                      28.3   
         CRM                                                        30.2   
         UCB                                                        10.9   
         SEEDA                                                      37.8   
         SEEDA (UCB)                                                 0.2   
         SEEDA Plateau (Paper)                                       0.0   
         SEEDA Plateau (UCB)                                         3.2   
         SEEDA Plateau (Two-sided, decoupled)                       16.6   
         SEEDA Plateau (Two-sided, decoupled, NoLog)                16.6   

                                                      Correct dose alloc %  
Scenario Algorithm                                                          
a = 1.0  3 + 3                                                       27.12  
         CRM                                                         29.58  
         UCB                                                         14.96  
         SEEDA                                                       25.80  
         SEEDA (UCB)                                                 21.40  
         SEEDA Plateau (Paper)                                       21.15  
         SEEDA Plateau (UCB)                                         23.58  
         SEEDA Plateau (Two-sided, decoupled)                        23.56  
         SEEDA Plateau (Two-sided, decoupled, NoLog)                 22.90

In [19]:
# Paper Table 2 layout (Recommended | Allocated side by side, mean over two lines
# with (std) beneath). Optimal biological dose (Dose 3) highlighted.
table2_real = build_table2(
    a_algo_rec_map, a_algo_alloc_map, a_key(A_STAR), ALGOS,
    N_REAL_COHORTS, N_LEVELS, TOXICITY_PROBS, EFFICACY_PROBS, n_batches=5,
)
print(f"Table 2 (realistic, n = {N_REAL_COHORTS} cohorts). "
      f"Optimal biological dose = Dose {OPTIMAL_DOSE + 1}.")
display(style_table2(table2_real, OPTIMAL_DOSE))

Table 2 (realistic, n = 10 cohorts). Optimal biological dose = Dose 3.


In [20]:
# Export the realistic Table 2 to LaTeX:
export_table2_latex(
    table2_real, OPTIMAL_DOSE,
    f"plots/{timestamp}/Realistic/real_table2.tex",
    caption=f"Recommendation \\& allocation percentages (realistic, "
            f"n = {N_REAL_COHORTS} cohorts). "
            f"Green column: optimal biological dose (Dose {OPTIMAL_DOSE + 1}); "
            f"the toxicity MTD is Dose {TOX_MTD + 1}. "
            f"\\textbf{{Bold}}: majority recommended or allocated dose per algorithm. "
            f"Mean over {N_TRIALS} repetitions, (std).",
    label="tab:real_table2"
)

Saved LaTeX table to plots/2026-07-14_21-04-53/Realistic/real_table2.tex


'plots/2026-07-14_21-04-53/Realistic/real_table2.tex'

#### Asymptotic

In [21]:
N_ASYM_COHORTS = 300

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [22]:
# Reinitialise the maps for the asymptotic setting:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_ASYM_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_safe_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_ASYM_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

# Run the asymptotic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0], n_cohorts=N_ASYM_COHORTS)
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1], n_cohorts=N_ASYM_COHORTS)
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2], n_cohorts=N_ASYM_COHORTS)
        
        # SEEDA (current version):
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA (UCB):
        seeda_original_dose_escalator = SEEDAOriginalDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_original_dose_escalator, a, ALGOS[4], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (Paper) - faithful first-flat-pair L1:
        seedapl_paper_dose_escalator = SEEDAPlateauNaiveDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_paper_dose_escalator, a, ALGOS[5], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (UCB) - entire-upper-tail-flat L1:
        seedapl_ucb_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_ucb_dose_escalator, a, ALGOS[6], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (Two-sided, decoupled) - UCB tail L1, no Part B, small L1 c:
        seedapl_twosided_dose_escalator = SEEDAPlateauTwoSidedDecoupledDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            l1_coefficient=L1_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_twosided_dose_escalator, a, ALGOS[7], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        # SEEDA Plateau (Two-sided, decoupled, NoLog) - same but no-log exploration:
        seedapl_twosided_nolog_dose_escalator = SEEDAPlateauTwoSidedDecoupledNoLogDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            ucb_coefficient=SEEDA_UCB_COEFF,
            l1_coefficient=L1_COEFF,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_twosided_nolog_dose_escalator, a, ALGOS[8], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [23]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_algo_dose_rec_proposals.png"
)

Plot the distribution of dose allocations, pooled across all cohorts and trial runs.

In [24]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_ASYM_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Asymptotic/Allocations/asym_algo_dose_allocations.png"
)

Plot the distribution of dose limiting events across all the trial runs and cohorts.

In [25]:
plot_n_dles(
    a_algo_n_dle_map, 
    img_path=f"plots/{timestamp}/Asymptotic/asym_algo_n_dles.png"
)

Plot the proposal accuracy (whether it matches the MTD) for each cohort so that we can see the time evolution of our `DoseEscalator`'s proposals. 

This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [26]:
plot_acc_progression(
    N_ASYM_COHORTS, 
    cohort_algo_rec_map,
    CORRECT_DOSES,
    show_fig=False, 
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_dose_rec_proposal_acc_progression.png"
)

Sample efficiency (Figure 3 in the SEEDA paper): the minimum number of patients needed to reach a given recommendation accuracy. This is the inverse of the accuracy-progression curve above. Conforms to the paper's definitions: recommendation accuracy is the successful-recommendation probability $P[\hat{d}(n) = k*]$ (Section 2.2), reproducing the sample-efficiency study of Section 5.1.3 (it inverts the mean accuracy curve rather than running the paper's exact per-trial early-stopping rule).

In [27]:
plot_sample_efficiency(
    N_ASYM_COHORTS,
    COHORT_SIZE,
    cohort_algo_rec_map,
    CORRECT_DOSES,
    algos=[a for a in ALGOS if a not in ("3 + 3", "CRM")],
    show_fig=False,
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_sample_efficiency.png"
)

Efficacy per patient and safety-violation percentage (Figure 2 in the SEEDA paper), versus number of cohorts. Conforms to the paper's definitions: efficacy per patient is the "effective treatment" objective: cumulative efficacy per patient (Section 2.2), using $E[X_t] = q_{I(t)}$ (the observed curve in expectation over trials); safety violation = % of trials whose running average true toxicity of allocated doses exceeds $\theta$, i.e. the safety constraint of Section 2.2 / Eq. (2) proven for the average true toxicity in Theorem 2. Both are computed from the pooled per-cohort allocations.

In [28]:
plot_efficacy_and_violation(
    N_ASYM_COHORTS,
    N_TRIALS,
    a_algo_alloc_map,
    EFFICACY_PROBS,
    TOXICITY_PROBS,
    TTL,
    img_path=f"plots/{timestamp}/Asymptotic/asym_efficacy_and_violation.png"
)

Type I (false alarm) and Type II (miss detection) error rates versus number of cohorts (Figure 1 in the SEEDA paper). Conforms to the paper's definitions (supplementary Section J): $e_1 = \sum_k 1{p_k ≤ \theta}·1{\hat{p}_k(n) > \theta}$ (Type I / false alarm) and $e_2 = \sum_k 1{p_k > \theta}·1{\hat{p}_k(n) ≤ \theta}$ (Type II / miss detection) — exactly the events of Lemma 1 and Lemma 2. The "declared safe" test $\hat{p}_k(n) ≤ \theta$ is the admissible set $\{k : p_k(\hat{a} + \alpha_t) ≤ \theta \}$ for SEEDA/Plateau, and estimated toxicity ≤ $\theta$ for UCB/CRM. 3 + 3 has no safety model and is skipped.

In [29]:
plot_error_rates(
    N_ASYM_COHORTS,
    cohort_algo_safe_map,
    TOXICITY_PROBS,
    TTL,
    show_fig=False,
    img_path=f"plots/{timestamp}/Asymptotic/asym_error_rates.png"
)

Make a table with the metrics.

In [30]:
# Long-format (scenario, algorithm, dose) table feeding the correct-dose summary below.
# Table 2 further down uses the raw maps directly, not this table.
asym_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, OPTIMAL_DOSES, N_LEVELS, ALGOS
)

correct_summary = build_correct_dose_summary(asym_table, CORRECT_MTDS)
print("Correct-dose summary")
display(correct_summary)

Correct-dose summary


Correct dose  Is MTD  \
Scenario Algorithm                                                           
a = 1.0  3 + 3                                                   3   False   
         CRM                                                     3   False   
         UCB                                                     3   False   
         SEEDA                                                   3   False   
         SEEDA (UCB)                                             3   False   
         SEEDA Plateau (Paper)                                   3   False   
         SEEDA Plateau (UCB)                                     3   False   
         SEEDA Plateau (Two-sided, decoupled)                    3   False   
         SEEDA Plateau (Two-sided, decoupled, NoLog)             3   False   

                                                      Correct dose rec %  \
Scenario Algorithm                                                         
a = 1.0  3 + 3                                                      27.5   
         CRM                                                        33.3   
         UCB                                                        19.0   
         SEEDA                                                      47.5   
         SEEDA (UCB)                                                 0.0   
         SEEDA Plateau (Paper)                                       0.0   
         SEEDA Plateau (UCB)                                         0.0   
         SEEDA Plateau (Two-sided, decoupled)                       93.1   
         SEEDA Plateau (Two-sided, decoupled, NoLog)                94.4   

                                                      Correct dose alloc %  
Scenario Algorithm                                                          
a = 1.0  3 + 3                                                       27.48  
         CRM                                                         33.24  
         UCB                                                         23.77  
         SEEDA                                                       42.72  
         SEEDA (UCB)                                                 27.23  
         SEEDA Plateau (Paper)                                       42.46  
         SEEDA Plateau (UCB)                                         42.86  
         SEEDA Plateau (Two-sided, decoupled)                        42.55  
         SEEDA Plateau (Two-sided, decoupled, NoLog)                 27.99

In [31]:
# Paper Table 2 layout (Recommended | Allocated side by side, mean over two lines
# with (std) beneath). Optimal biological dose (Dose 3) highlighted.
table2_asym = build_table2(
    a_algo_rec_map, a_algo_alloc_map, a_key(A_STAR), ALGOS,
    N_ASYM_COHORTS, N_LEVELS, TOXICITY_PROBS, EFFICACY_PROBS, n_batches=5,
)
print(f"Table 2 (asymptotic, n = {N_ASYM_COHORTS} cohorts). "
      f"Optimal biological dose = Dose {OPTIMAL_DOSE + 1}.")
display(style_table2(table2_asym, OPTIMAL_DOSE))

Table 2 (asymptotic, n = 300 cohorts). Optimal biological dose = Dose 3.


In [32]:
# Export the asymptotic Table 2 to LaTeX:
export_table2_latex(
    table2_asym, OPTIMAL_DOSE,
    f"plots/{timestamp}/Asymptotic/asym_table2.tex",
    caption=f"Recommendation \\& allocation percentages (asymptotic, "
            f"n = {N_ASYM_COHORTS} cohorts). "
            f"Green column: optimal biological dose (Dose {OPTIMAL_DOSE + 1}); "
            f"the toxicity MTD is Dose {TOX_MTD + 1}. "
            f"\\textbf{{Bold}}: majority recommended or allocated dose per algorithm. "
            f"Mean over {N_TRIALS} repetitions, (std).",
    label="tab:asym_table2"
)

Saved LaTeX table to plots/2026-07-14_21-04-53/Asymptotic/asym_table2.tex


'plots/2026-07-14_21-04-53/Asymptotic/asym_table2.tex'